# TV3.1 — Giao thức điền khuyết trên dữ liệu USGS

Notebook đọc bảng silver qua hàm read_silver của nhóm, che các ô đã quan sát,
chạy hai baseline có sẵn rồi tính sai số trên các ô bị che. Cả hai baseline
dùng cùng mask trong mỗi kịch bản. Các giá trị thiếu sẵn không có đáp án để chấm.

Mặc định dùng 10 trạm chuẩn và nửa cuối năm 2025 theo data contract.
Chạy từ thư mục repo hoặc notebooks/. Nếu silver nằm ở data lake khác,
đặt WQ_LAKE_PATH. Để minh họa trên một cửa sổ ngắn hơn, đặt
WQ_TV3_START và WQ_TV3_END theo dạng YYYY-MM-DD. Mốc kết thúc không được tính.

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "config" / "stations.json").is_file():
    raise RuntimeError("Mở notebook từ thư mục gốc của repo hoặc notebooks/.")
os.chdir(root)
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from wq.contracts import CORE_VARIABLES, SPLITS
from wq.evaluation import apply_mask, make_mask, score_imputation
from wq.io import read_silver
from wq.models.baselines import LOCFImputer, LinearInterpolationImputer

In [ ]:
variables = CORE_VARIABLES
rates = (0.05, 0.10, 0.20, 0.30)
kinds = ("random", "block")
seed = 42
block_steps = (4, 96)  # 1–24 giờ trên lưới 15 phút

test_start, test_end = SPLITS["test"]
start = os.environ.get("WQ_TV3_START", test_start)
end = os.environ.get("WQ_TV3_END", test_end)
truth = read_silver(stations="core", start=start, end=end)
if truth.empty or not truth[variables].notna().any().any():
    raise ValueError("Không có giá trị đo trong cửa sổ silver đã chọn.")

observed = truth[variables].notna()
natural_missing = 1 - observed.mean()
print(f"{len(truth):,} dòng, {truth['station_id'].nunique()} trạm, [{start}, {end}) UTC")
display(natural_missing.rename("tỷ lệ thiếu tự nhiên").mul(100).round(2).to_frame())

## Tạo mask, chạy baseline và chấm điểm

n_masked là số ô có đáp án đã bị che. n_scored là số ô trong đó mô hình trả
về giá trị hữu hạn. Nếu hai mô hình có độ bao phủ khác nhau, đọc độ bao phủ
cùng với MAE, RMSE và R² vì sai số có thể được tính trên các tập ô khác nhau.

In [ ]:
models = (LOCFImputer(), LinearInterpolationImputer())
scores = []
realized_rates = []
example_kind = kinds[-1]
example_rate = 0.20 if 0.20 in rates else rates[-1]
example = {}

for kind in kinds:
    for rate in rates:
        mask = make_mask(
            truth, rate, kind, variables=variables, block_steps=block_steps, seed=seed
        )
        masked = apply_mask(truth, mask)
        assert not (mask & ~observed).to_numpy().any()
        assert masked[variables].isna().equals(~observed | mask)

        n_observed = observed.sum()
        assert mask.sum().equals((n_observed * rate).round().astype(int))
        realized_rates.append(
            (mask.sum() / n_observed.where(n_observed > 0))
            .rename((kind, rate))
        )

        for model in models:
            imputed = model.transform(masked)
            result = score_imputation(truth, imputed, mask, variables)
            scores.append(result.assign(method=model.name, missing_type=kind, missing_rate=rate))
            if (kind, rate) == (example_kind, example_rate):
                example[model.name] = imputed

        if (kind, rate) == (example_kind, example_rate):
            example_mask = mask

actual_rates = pd.DataFrame(realized_rates).mul(100).round(2)
actual_rates.index = pd.MultiIndex.from_tuples(
    actual_rates.index, names=["kiểu che", "tỷ lệ yêu cầu"]
)
display(actual_rates)

results = pd.concat(scores, ignore_index=True)
results["coverage"] = results["n_scored"] / results["n_masked"].where(results["n_masked"] > 0)
display(results.round(4))

## Đối chiếu giá trị thật và giá trị điền

Biểu đồ hiển thị các giá trị được điền tại những ô đã che trong kịch bản minh họa,
ưu tiên DO nếu biến đó nằm trong danh sách cấu hình.

In [ ]:
variable = "dissolved_oxygen" if "dissolved_oxygen" in variables else variables[0]
masked_rows = truth.loc[example_mask[variable], ["station_id", "ts"]]
if masked_rows.empty:
    print(f"Không có ô bị che để vẽ cho {variable}.")
else:
    station = masked_rows["station_id"].value_counts().idxmax()
    masked_times = masked_rows.loc[masked_rows["station_id"].eq(station), "ts"]
    center = masked_times.iloc[len(masked_times) // 2]
    window = truth["station_id"].eq(station) & truth["ts"].between(
        center - pd.Timedelta(hours=24), center + pd.Timedelta(hours=24)
    )
    points = window & example_mask[variable]
    fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
    ax.plot(truth.loc[window, "ts"], truth.loc[window, variable], label="Giá trị thật")
    for method, frame in example.items():
        ax.scatter(frame.loc[points, "ts"], frame.loc[points, variable], s=16, label=method)
    ax.set(
        title=f"{station} · {example_kind} · tỷ lệ che {example_rate:.0%}",
        xlabel="Thời gian (UTC)",
        ylabel=variable,
    )
    ax.legend()
    plt.show()